In [1]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import scipy
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
import numpy as np

from statsmodels.discrete.conditional_models import ConditionalLogit

from Preproces_prod2 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data, flexible_matching, summary_inicial, comparar_medias_test,analyze_vrs_data, match_nn_funcionable, integer_programming_matching_gurobi, match_nn, match_nn_max_dist,match_nn_max_dist_weigths

import inv

In [2]:
from IPython.core.display import display
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)


In [9]:
path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [12]:
df_all = call_data('NAC_RNI_EGRESOS_ENTREGA_ISCI_28_10_2024_encrip.csv',upc=True)

174441
post merge comunas 174441
['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
post merge rural 174441
Datos perdidos por muertes:  1135
Datos perdidos por filtro peso:  2215
Datos perdidos por filtro semanas y peso:  2755
reemplazos n/a net inmunizado:  24
Datos perdidos por fecha ingreso menor a fecha nacimiento: 0
Datos perdidos

In [13]:
#CHILE CHICO
df = (
    df_all
    .query('(FECHA_ING > "2024-03-31") | FECHA_ING.isna()')
    .query('chile_chico==1')
    .copy()
)

# scaler = StandardScaler()
# df['PESO_scl'] = scaler.fit_transform(df[['PESO']])

# scaler = StandardScaler()
# df['edad_relativa_scl'] = scaler.fit_transform(df[['edad_relativa']])

# scaler = StandardScaler()
# df['ingreso_relativo_scl'] = scaler.fit_transform(df[['ingreso_relativo']])

# scaler = StandardScaler()
# df['SEMANAS_scl'] = scaler.fit_transform(df[['SEMANAS']])

In [8]:
df_vrs_match_case = df.copy()

In [14]:
df_upc_match_case = df.copy()

In [ ]:
df_vrs_match_case.to_csv(path_data/'df_vrs_match_case_2810_clchico.csv') #, index=False

In [15]:
df_upc_match_case.to_csv(path_data/'df_upc_match_case_2810_clchico.csv')

In [ ]:
df.shape

In [ ]:
summary_inicial(df)

In [5]:
np.random.seed(42)
dict_experiments={}

In [6]:
#PARTE MANUAL 
list_experiments=[]

# Sección 1: usar toda la muestra

Primer experimento: hacemos matching x ESTABLECIMEINTO

Segundo experimento: hacemos matching x COMUNA


In [16]:
df = df_vrs_match_case.copy()

In [17]:
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 680 y control 106183


In [ ]:
#Primer experimento: hacemos matching x ESTABLECIMEINTO
# Paso 1: Aplicar el matching con opciones de intervalos
intervals = {'PESO': 300}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION','COMUNA'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'FECHA_NAC': 28} 
# Aplicamos la función de matching con la variable de fecha
matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)

In [ ]:
columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
#incluye todo
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
results['method']='MIP'
results['subset']='S1'
results['region']='all'
list_experiments.append(results.copy())


In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','SEXO', 'PESO'], 
                                                              match_vars_exact = ['SEXO'],
                                                              match_vars_onehot=['NOMBRE_REGION'],
                                                              ratio="1:1")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
results['method']='KNN'
results['subset']='S1'
results['region']='all'
list_experiments.append(results.copy())

In [ ]:
# Lista para almacenar los DataFrames de cada experimento con el nombre del experimento
experiment_dfs = []

for result in list_experiments:
    # Obtener el DataFrame de resultados
    results_df_transposed =(
        result['results_df'].T
        .assign(
            str_values = lambda df : f'{str(df["Efectividad"].values[0].round(2))} ({str(df["0.025 Conf Interval"].values[0].round(2))}; {str(df["0.975 Conf Interval"].values[0].round(2))})'
        )
        .assign(
                method = result['method'],
                subset = result['subset'],
                region = result['region']
            )
        .pipe(lambda df:df.set_index(['region','method','subset']))
        ) 
    
    # Añadir el DataFrame a la lista
    experiment_dfs.append(results_df_transposed)

# Concatenar todos los DataFrames en uno solo
all_results_df = pd.concat(experiment_dfs, ignore_index=False)

# Mostrar el DataFrame concatenado
display(all_results_df)




marcel_summary= (all_results_df[['str_values']]
#.reset_index()
.unstack(-2)
.unstack(-1)

)

display(marcel_summary)

In [ ]:
dict_experiments['Experimento_vrs_notvrs_nn']=results

## Segundo experimento: hacemos matching x COMUNA

In [ ]:
#Primer experimento: hacemos matching x ESTABLECIMEINTO
# Paso 1: Aplicar el matching con opciones de intervalos
intervals = {'PESO': 300}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION','COMUNA_N'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'FECHA_NAC': 28} 
# Aplicamos la función de matching con la variable de fecha
matched_data,matched_controls = flexible_matching(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])


In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn_funcionable(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl', 'PESO_scl'], 
                                                              match_vars_exact = ['SEXO','COMUNA'],
                                                              match_vars_onehot=[],
                                                              ratio="1:1")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO_scl','SEXO','edad_relativa_scl','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
intervals = {'PESO': 300}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','COMUNA'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'FECHA_NAC': 28} 
# Aplicamos la función de matching con la variable de fecha
matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO_scl','SEXO','edad_relativa_scl','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
dict_experiments['Experimento_vrs_novrs_comuna']=results

# Sección 2: usar Egresos y enfermedades

Dianosticos x letra
P    7090: Ciertas afecciones originadas en el período perinatal

J    1865: Enfermedades del sistema respiratorio 

Q     636: Malformaciones congénitas, deformidades y anomalías cromosómicas

Z     459: Factores que influyen en el estado de salud y contacto con los  servicios de salud

N     356: Enfermedades del sistema genitourinario

A     292: Ciertas enfermedades infecciosas y parasitarias 

K     249: Enfermedades del sistema digestivo

R     199: Síntomas, signos y hallazgos anormales clínicos y de laboratorio, no clasificados en otra parte

G     157: Enfermedades del sistema nervioso central

S      93: Traumatismos, envenenamientos y algunas otras consecuencias de causas externas

En esta ocasion usaramoes A, pero vamos a tener pocos casos
Primer experimento: hacemos matching x ESTABLECIMEINTO
Segundo experimento: hacemos matching x COMUNA


In [ ]:
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False & diag_1_leter=='K'").copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

In [ ]:
intervals = {'PESO': 300}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION'] #[,'COMUNA_N' 'ESTAB','COMUNA_N'
match_date_vars =  {'FECHA_NAC': 28, 'FECHA_ING_any':15} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])


In [ ]:
intervals = {}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['ESTAB','Group'] #[,'COMUNA_N' 'ESTAB','COMUNA_N'
match_date_vars =  {'FECHA_NAC': 28, 'FECHA_ING_any':15} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)

columns_keep= ['diag_vrs','estado_inmunizacion','Group','SEXO','PESO','SEMANAS', 'NOMBRE_REGION','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
# print(f'\n {len(set(np.concatenate([i for i in feasible_controls.values() if len(i)>0])))}')

In [ ]:
results = analyze_vrs_data(df_matched,covs=['SEXO','PESO','SEMANAS'])

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn_funcionable(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl', 'PESO_scl','ingreso_relativo'], 
                                                              match_vars_exact = ['SEXO','COMUNA'],
                                                              match_vars_onehot=[],
                                                              ratio="2:1")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO_scl','SEXO','edad_relativa_scl','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:

matched_data, matched_incompleto, matched_controls = match_nn_max_dist(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl','ingreso_relativo_scl'], 
                                                              match_vars_exact = ['SEXO','NOMBRE_REGION','SEMANAS'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="2:1",
                                                              max_distance=20)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','edad_relativa_scl','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])


In [ ]:
def apply_weights(df, columns, weights):
    weighted_df = df.copy()
    for col in columns:
        weighted_df[col] *= weights.get(col, 1) 
    return weighted_df

match_vars_nn = ['edad_relativa_scl', 'PESO_scl']
weights = {'edad_relativa_scl': 1.5, 'PESO_scl': 0.8} 


scaler = StandardScaler()
covariables_case_scaled = pd.DataFrame(scaler.fit_transform(covariables_case[match_vars_nn]), columns=match_vars_nn)
covariables_control_scaled = pd.DataFrame(scaler.transform(covariables_control[match_vars_nn]), columns=match_vars_nn)

covariables_case_weighted = apply_weights(covariables_case_scaled, match_vars_nn, weights)
covariables_control_weighted = apply_weights(covariables_control_scaled, match_vars_nn, weights)

nn = NearestNeighbors(n_neighbors=5, algorithm='auto')
nn.fit(covariables_control_weighted)

# Calcular los vecinos más cercanos para un caso de ejemplo
distances, indices = nn.kneighbors([covariables_case_weighted.iloc[0]])

# Seccion 3 
 Control patients were defined as infants younger than 12 months of age who had visited the pediatric emergency department of a partici- pating center for one of the following diagnoses: urinary tract infection without respiratory symp- toms, acute gastroenteritis without respiratory symptoms, infantile colic without fever or respira- tory symptoms, weight loss or feeding difficulties without fever or respiratory symptoms, neonatal jaundice without fever or respiratory symptoms, abnormal crying (based on parental subjectivi- ty) without fever or acute respiratory symptoms, head trauma without fever or respiratory symp- toms, or emergency surgery without fever or re- spiratory symptoms. 
 
los códigos ICD-10 que corresponden a las condiciones mencionadas en el texto:

1. **Infección del tracto urinario (sin síntomas respiratorios)**:  
   - N39.0 (Infección del tracto urinario, sitio no especificado)

2. **Gastroenteritis aguda (sin síntomas respiratorios)**:  
   - A09 (Gastroenteritis y colitis de origen infeccioso no especificado)

3. **Cólico infantil (sin fiebre ni síntomas respiratorios)**:  
   - R10.83 (Cólico del lactante)

4. **Pérdida de peso o dificultades de alimentación (sin fiebre ni síntomas respiratorios)**:  
   - R63.4 (Pérdida de peso)
   - R63.3 (Dificultades de alimentación y alimentación deficiente)

5. **Ictericia neonatal (sin fiebre ni síntomas respiratorios)**:  
   - P59.9 (Ictericia neonatal no especificada)

6. **Llanto anormal (sin fiebre ni síntomas respiratorios)**:  
   - R68.11 (Llanto excesivo del lactante)

7. **Traumatismo craneal (sin fiebre ni síntomas respiratorios)**:  
   - S09.90 (Traumatismo de la cabeza, no especificado)

8. **Cirugía de emergencia (sin fiebre ni síntomas respiratorios)**:  
   - Z53.9 (Procedimiento quirúrgico no realizado por otra razón)




In [ ]:
icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
]


In [ ]:
df_case= df.query("diag_vrs==True").copy()
df_control= df[df.DIAG1.isin(icd10_codes_fr)].copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

In [ ]:
#Primer experimento: hacemos matching x ESTABLECIMEINTO
# Paso 1: Aplicar el matching con opciones de intervalos
intervals = {}  # Por ejemplo, 5 kilos y 3 días de intervalo    'PESO': 300, 'SEMANAS': 1
match_vars = ['NOMBRE_REGION','group'] #[,'COMUNA_N' 'ESTAB','SEMANAS' ,'PESO','NOMBRE_REGION' 'SEXO',
match_date_vars =  {'FECHA_NAC': 28, 'FECHA_ING_any':15} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                       df_control=df_control,
                                       match_vars=match_vars, # inútil
                                       intervals=intervals,
                                       match_date_vars=match_date_vars, 
                                       n_control=1)

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','SEXO','SEMANAS','PESO', 'DIAG1','diag_1_leter','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
scaler = StandardScaler()
df_matched['PESO_scl'] = scaler.fit_transform(df_matched[['PESO']])

In [ ]:
results = analyze_vrs_data(df_matched,covs=['SEXO','SEMANAS','PESO_scl'])

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
endog = df_matched['diag_vrs']

exog = df_matched[['estado_inmunizacion','SEXO','SEMANAS','PESO_scl']]

model = ConditionalLogit(endog, exog, groups = df_matched['Group'])

# Ajustar el modelo con optimización BFGS y parámetros iniciales
result = model.fit(method='BFGS', start_params=[0] * exog.shape[1])

# Imprimir el resumen del modelo
print(result.summary())

In [ ]:
odds_ratios = (
        pd.DataFrame({
            'Coeficientes': result.params,
            'OR': np.exp(result.params),
            'Efectividad': (1 - np.exp(result.params)) * 100,
        })
        .merge(result.conf_int().rename({0:'0.025', 1: '0.975'}, axis=1), left_index=True, right_index=True)
    )

In [ ]:
len(set(np.concatenate([i for i in feasible_controls.values() if len(i)>0])))

In [ ]:
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
dict_experiments['Experimento_3_1']=results

In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn_funcionable(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl', 'PESO_scl','ingreso_relativo'], 
                                                              match_vars_exact = ['SEXO','NOMBRE_REGION'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="1:1")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO_scl','SEXO','edad_relativa_scl','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

## not working change groups

In [ ]:
# Función para obtener el valor de la segunda fila en la columna 'DIAG1'
def get_second_diag1(group):
    if len(group) > 1:
        # Si el grupo tiene al menos dos filas, devolver el valor de la segunda fila
        return group.iloc[1]['DIAG1']
    else:
        # Si el grupo tiene solo una fila, devolver NaN (o algún valor por defecto)
        return np.nan

In [ ]:
#prueba de codido
import pandas as pd
import numpy as np
from statsmodels.discrete.conditional_models import ConditionalLogit

# Transformar 'diag_1_leter' a variables dummy
df_matched_dummies = pd.get_dummies(df_matched, columns=['diag_1_leter'], drop_first=True,dtype='int')

# Definir las variables endógena y exógena
endog = df_matched_dummies['diag_vrs']
# Agregar las dummies de 'diag_1_leter' a las variables exógenas
exog = pd.concat([df_matched_dummies[['estado_inmunizacion']], df_matched_dummies.filter(like='diag_1_leter_')], axis=1)

# Ajustar modelo de regresión logística condicional
model = ConditionalLogit(endog, exog, groups=df_matched_dummies['Group'])

# Ajustar el modelo con optimización BFGS y parámetros iniciales
result = model.fit(method='BFGS', start_params=[0] * exog.shape[1])

# Imprimir el resumen del modelo
print(result.summary())


In [ ]:
df_matched_dummi = df_matched.copy()
second_diag1_per_group = df_matched_dummi.groupby('Group').apply(get_second_diag1)
df_matched_dummi['id_group'] = df_matched_dummi['Group'].map(second_diag1_per_group)

In [ ]:
# Llamada a la función
results = analyze_vrs_data(df_matched_dummi,group_col='id_group')

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
dict_experiments['Experimento_3_2']=results

In [ ]:
# # Ajustar modelo de regresión logística condicional
# endog = df_matched['diag_vrs']
# exog = df_matched[['estado_inmunizacion']]
# model = ConditionalLogit(endog, exog, groups=df_matched['Group'])

# #result = model.fit()
# result = model.fit(method='BFGS', start_params=[0] * exog.shape[1])




# # Crear DataFrame con coeficientes, odds ratios y efectividad
# odds_ratios = (
#     pd.DataFrame({
#         'Coeficientes': result.params,
#         'OR': np.exp(result.params),
#         'Efectividad': (1 - np.exp(result.params)) * 100,
#     })
#     .merge(result.conf_int().rename({0:'0.025', 1: '0.975'}, axis=1), left_index=True, right_index=True)
# )

# odds_ratios

# Seccion 4
 Muy prematuros all chile

In [ ]:
icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
]

In [ ]:
df_case= df.query("(diag_vrs==True) & (muy_prematuro==1)").copy()
df_control= df.query("(muy_prematuro==1) & (diag_vrs==False)").copy() # DIAG1.isin(@icd10_codes_fr)
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

In [ ]:
intervals = {}  #
match_vars = [] #[,'COMUNA_N' 'ESTAB','NOMBRE_REGION', ,'PESO', NOMBRE_REGION
match_date_vars =  {'FECHA_NAC': 28, 'FECHA_ING_any':15} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                       df_control=df_control,
                                       match_vars=match_vars, # inútil
                                       intervals=intervals,
                                       match_date_vars=match_date_vars, 
                                       n_control=1)

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','DIAG1','diag_1_leter','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()


In [ ]:
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn_funcionable(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl', 'PESO_scl','ingreso_relativo'], 
                                                              match_vars_exact = ['SEXO','NOMBRE_REGION'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="1:2")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO_scl','SEXO','edad_relativa_scl','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

# Seccion 5 
    REGIONES RARAS

In [ ]:
df_case= df.query('(diag_vrs==True) & (NOMBRE_REGION=="COQUIMBO")').copy()
df_control= df.query('DIAG1.isin(@icd10_codes_fr) & (NOMBRE_REGION=="COQUIMBO")').copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

In [ ]:
intervals = {}  #'PESO': 1000
match_vars = ['NOMBRE_REGION'] #[,'COMUNA_N' 'ESTAB','NOMBRE_REGION'
match_date_vars =  {'FECHA_NAC': 28, 'FECHA_ING_any':15}

matched_data,matched_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals,
                                 n_control=2
                                 )

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'NOMBRE_REGION','DIAG1','diag_1_leter','edad_relativa','ingreso_relativo']
df_matched = (matched_data[columns_keep])
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()

In [ ]:
comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','edad_relativa','ingreso_relativo'])

In [ ]:
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
matched_data, matched_incompleto, matched_controls = match_nn_max_dist(df_control, df_case,
                                                              match_vars_nn=['edad_relativa_scl','ingreso_relativo_scl'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="1:1",
                                                              max_distance=20)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','edad_relativa_scl','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])


In [ ]:
weights = {'edad_relativa': 1.5,'ingreso_relativo':10}

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','ingreso_relativo'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="1:1",
                                                              max_distance=10,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','edad_relativa_scl','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

In [ ]:
df_case= df.query('(diag_vrs==True) & (NOMBRE_REGION=="ANTOFAGASTA")').copy()
df_control= df.query('(diag_vrs==False)  & (NOMBRE_REGION=="ANTOFAGASTA")').copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

In [ ]:
weights = {'edad_relativa': 1.5} #,'ingreso_relativo':10

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa'],  #,'ingreso_relativo'
                                                              match_vars_exact = ['SEXO','SEMANAS','group'],
                                                              match_vars_onehot=['COMUNA'],
                                                              ratio="1:3",
                                                              max_distance=0.5#,
                                                             # weights=weights
                                                             )

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa'])) #,'ingreso_relativo'

columns_keep= ['diag_vrs','estado_inmunizacion','Group', 'SEXO','NOMBRE_REGION','edad_relativa','edad_relativa_scl','ingreso_relativo','PESO','PESO_scl']
df_matched = (matched_data[columns_keep])
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])